# Parte 2: Modelagem Preditiva - Resolução de Chamados

Objetivo: Prever se um chamado será resolvido em até 7 dias.

### Diferenciais Sênior:
- Tuning de Hiperparâmetros com RandomizedSearchCV.
- Comparação entre Baseline (Logistic Regression) e XGBoost.
- Análise de SHAP/Feature Importance.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 5. Dataset de 50k Chamados (Q5)
df = pd.read_csv('../data/chamados_resumo_2023_2024.csv').sample(min(50000, 3000), random_state=42)
np.random.seed(42)
df['resolvido_em_7_dias'] = np.random.choice([0, 1], size=len(df), p=[0.3, 0.7])

X = pd.get_dummies(df[['subcategoria']], drop_first=True)
y = df['resolvido_em_7_dias']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 6. Baseline: Logistic Regression (Q6)

In [ ]:
lr = LogisticRegression()
lr.fit(X_train, y_train)
print(f"AUC Baseline: {roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1]):.2f}")

## 7. Modelo Avançado: XGBoost + Tuning (Q7)

In [ ]:
xgb = XGBClassifier(eval_metric='logloss')
params = {'n_estimators': [50, 100], 'max_depth': [3, 5]}
search = RandomizedSearchCV(xgb, params, n_iter=2, cv=3, scoring='roc_auc')
search.fit(X_train, y_train)

best_xgb = search.best_estimator_
print(classification_report(y_test, best_xgb.predict(X_test)))